# Set up workspace

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr

import time
import s3fs

from srm import catalog

# Define parameters

In [2]:
from srm import run_bcsd

# Run through clunky BCSD workflow

In [3]:
# Rechunking in between steps
dict_all = run_bcsd.main()

Loaded data: 1.44 seconds
Subset time: 0.09 seconds
Rechunked obs to full space: 52.31 seconds
Interpolated obs to coarse grid: 0.07 seconds
Rechunked all to full time: 17.23 seconds


100%|██████████| 55296/55296 [01:04<00:00, 852.36it/s]


Quantile mapped historical: 72.75 seconds


100%|██████████| 55296/55296 [02:07<00:00, 435.21it/s]


Quantile mapped future: 140.01 seconds
Calculate error map for spatial disaggregation: 153.45 seconds
Downscaled historical: 252.67 seconds
Downscaled future: 644.21 seconds
TOTAL TIME: 1332.80 seconds


In [ ]:
# Not rechunking in between steps
dict_all = run_bcsd.main(rechunk_workflow=False)

Loaded data: 1.31 seconds
Subset time: 0.10 seconds
Interpolated obs to coarse grid: 127.59 seconds


# Look at results

In [ ]:
%%time
dict_all["model_hist_debiased_downscaled"]=dict_all["model_hist_debiased_downscaled"].chunk(time=1000,lat=7,lon=7)

In [ ]:
%%time
dict_all["ssp245_debiased_downscaled"]=dict_all["ssp245_debiased_downscaled"].chunk(time=1000,lat=7,lon=7)

In [ ]:
cities = gpd.read_file(
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_populated_places_simple.zip",
    columns=["name", "sov0name", "geometry"],
)
sa_cities = cities[cities["sov0name"] == "South Africa"]
cape_town_x, cape_town_y = (
    float(sa_cities[sa_cities["name"] == "Cape Town"].geometry.x),
    float(sa_cities[sa_cities["name"] == "Cape Town"].geometry.y),
)

In [ ]:
%%time
dict_all["model_hist_debiased_downscaled"].sel(lat=cape_town_y, lon=cape_town_x, method="nearest").rolling(time=365*5).mean().plot()
dict_all["ssp245_debiased_downscaled"].sel(lat=cape_town_y, lon=cape_town_x, method="nearest").rolling(time=365*5).mean().plot()